# **Notebook 3 - Basket Analysis**

Afonso Fernandes 20241710, Lourenço Lima 20241711, Lucas Casimiro 20241796

## Imports

In [37]:
import os
import sys
import warnings
from pathlib import Path
import pandas as pd

def _find_project_root(start, marker="requirements.txt"):
    path = Path(start).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / marker).exists():
            return str(candidate)
    raise RuntimeError(f"Could not find project root (marker={marker!r}, searched from {start})")

PROJECT_ROOT = _find_project_root(os.path.abspath("."))
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import math
from sklearn.preprocessing import RobustScaler

from functions.eda import *
from functions.preprocessing import *
from functions.basket import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **1. Merging data**

In [39]:
customer_basket = pd.read_csv('data/customer_basket.csv')
customer = pd.read_csv('data/ci_clustered.csv')

# **2. Dataframe by cluster**

In [40]:
cluster_dataframes = {
    cluster: group
    for cluster, group in customer.groupby("final_cluster_label")
}

loyal_core_spenders = cluster_dataframes.get('Loyal core spenders')
bargain_hunters = cluster_dataframes.get('Bargain hunters')
big_families = cluster_dataframes.get('Big families (big spenders)')
vegans = cluster_dataframes.get('Vegans')
tech_enthusiasts = cluster_dataframes.get('Tech enthusiasts')
karens = cluster_dataframes.get('Karens')
gamers = cluster_dataframes.get('Gamers')


## Parameter by cluster

In [41]:
params_by_cluster = {
    "Loyal core spenders": {
        "min_support": 0.02,
        "min_threshold": 1,
        "min_confidence": 0.10
    },
    "Vegans": {
        "min_support": 0.01,
        "min_threshold": 1,
        "min_confidence": 0.10
    },
    "Bargain hunters": {
        "min_support": 0.01,
        "min_threshold": 1,
        "min_confidence": 0.10
    },
    "Karens": {
        "min_support": 0.008,
        "min_threshold": 1,
        "min_confidence": 0.08
    },
    "Tech enthusiasts": {
        "min_support": 0.006,
        "min_threshold": 1,
        "min_confidence": 0.08
    },
    "Big families (big spenders)": {
        "min_support": 0.006,
        "min_threshold": 1,
        "min_confidence": 0.08
    },
    "Gamers": {
        "min_support": 0.004,
        "min_threshold": 1,
        "min_confidence": 0.06
    }
}

# **3. Association rules**

In [42]:
# 2. Reload do basket.py
import importlib
import functions.basket as basket_module

importlib.reload(basket_module)

generate_rules_for_all_clusters = basket_module.generate_rules_for_all_clusters
generate_association_rules = basket_module.generate_association_rules

In [ ]:
all_cluster_rules = generate_rules_for_all_clusters(
    cluster_dataframes=cluster_dataframes,
    basket=customer_basket,
    params_by_cluster=params_by_cluster
)

for cluster_name, rules in all_cluster_rules.items():
    print(f"CLUSTER: {cluster_name}")

    if rules.empty:
        print("No rules found.")
    else:
        display(rules.head(30))


CLUSTER: Bargain hunters


,antecedents,consequents,support,confidence,lift
0,"frozenset({energy drink, laptop})",frozenset({bluetooth headphones}),0.010753,0.458716,3.777499
1,"frozenset({protein bar, bluetooth headphones})",frozenset({energy drink}),0.010251,0.432024,3.497816
2,"frozenset({bluetooth headphones, laptop})",frozenset({energy drink}),0.010753,0.426136,3.450146
3,"frozenset({protein bar, energy drink})",frozenset({bluetooth headphones}),0.010251,0.414493,3.413326
4,frozenset({bluetooth headphones}),"frozenset({energy drink, airpods})",0.025591,0.210744,3.390860
5,"frozenset({energy drink, airpods})",frozenset({bluetooth headphones}),0.025591,0.411765,3.390860
6,"frozenset({protein bar, airpods})",frozenset({energy drink}),0.016774,0.417112,3.377084
7,frozenset({energy drink}),"frozenset({protein bar, airpods})",0.016774,0.135810,3.377084
8,"frozenset({samsung galaxy 10, airpods})",frozenset({energy drink}),0.010824,0.409214,3.313138
9,frozenset({energy drink}),"frozenset({bluetooth headphones, airpods})",0.025591,0.207197,3.310876



CLUSTER: Big families (big spenders)


,antecedents,consequents,support,confidence,lift
0,"frozenset({fresh bread, black tea})","frozenset({salt, cereals})",0.006843,0.111959,1.860202
1,"frozenset({salt, cereals})","frozenset({fresh bread, black tea})",0.006843,0.113695,1.860202
2,"frozenset({honey, milk})","frozenset({oatmeal, cereals})",0.007465,0.166090,1.857319
3,"frozenset({oatmeal, cereals})","frozenset({honey, milk})",0.007465,0.083478,1.857319
4,frozenset({vegetables mix}),"frozenset({eggs, honey})",0.006687,0.161049,1.842603
5,"frozenset({honey, cereals, milk})",frozenset({oatmeal}),0.007465,0.393443,1.835875
6,"frozenset({eggs, nonfat milk})",frozenset({oatmeal}),0.006843,0.392857,1.833143
7,"frozenset({salt, honey})",frozenset({oil}),0.007154,0.219048,1.824451
8,"frozenset({eggs, vegetables mix})",frozenset({honey}),0.006687,0.383929,1.816527
9,frozenset({vegetables mix}),"frozenset({butter, honey})",0.006376,0.153558,1.772672



CLUSTER: Gamers


...

,antecedents,consequents,support,confidence,lift
0,frozenset({eggs}),frozenset({cereals}),0.021959,0.236877,2.626763
1,frozenset({cereals}),frozenset({eggs}),0.021959,0.243511,2.626763
2,frozenset({fresh bread}),frozenset({cereals}),0.021497,0.221728,2.458774
3,frozenset({cereals}),frozenset({fresh bread}),0.021497,0.238388,2.458774
4,frozenset({cereals}),frozenset({butter}),0.021898,0.242828,2.380549
5,frozenset({butter}),frozenset({cereals}),0.021898,0.214674,2.380549
6,frozenset({butter}),frozenset({eggs}),0.022483,0.220411,2.377579
7,frozenset({eggs}),frozenset({butter}),0.022483,0.242525,2.377579
8,frozenset({airpods}),frozenset({energy drink}),0.027688,0.189144,2.336875
9,frozenset({energy drink}),frozenset({airpods}),0.027688,0.342085,2.336875



CLUSTER: Tech enthusiasts


,antecedents,consequents,support,confidence,lift
0,frozenset({energy drink}),"frozenset({airpods, gadget for tiktok streaming})",0.007690,0.093074,6.347165
1,"frozenset({airpods, gadget for tiktok streaming})",frozenset({energy drink}),0.007690,0.524390,6.347165
2,frozenset({bluetooth headphones}),"frozenset({brownies, airpods})",0.006438,0.097035,6.309534
3,"frozenset({brownies, airpods})",frozenset({bluetooth headphones}),0.006438,0.418605,6.309534
4,"frozenset({energy drink, airpods})",frozenset({gadget for tiktok streaming}),0.007690,0.226316,5.805311
5,frozenset({gadget for tiktok streaming}),"frozenset({energy drink, airpods})",0.007690,0.197248,5.805311
6,"frozenset({energy drink, gadget for tiktok str...",frozenset({airpods}),0.007690,0.661538,5.579673
7,"frozenset({energy drink, iphone 10})",frozenset({airpods}),0.009120,0.629630,5.310541
8,"frozenset({energy drink, airpods})",frozenset({bluetooth headphones}),0.011803,0.347368,5.235806
9,frozenset({bluetooth headphones}),"frozenset({energy drink, airpods})",0.011803,0.177898,5.235806



CLUSTER: Vegans


,antecedents,consequents,support,confidence,lift
0,"frozenset({babies food, napkins})",frozenset({cooking oil}),0.015739,0.357704,2.624588
1,frozenset({cooking oil}),"frozenset({babies food, napkins})",0.015739,0.115482,2.624588
2,frozenset({napkins}),"frozenset({dog food, cooking oil})",0.015843,0.112008,2.613426
3,"frozenset({dog food, cooking oil})",frozenset({napkins}),0.015843,0.369653,2.613426
4,frozenset({napkins}),"frozenset({babies food, cooking oil})",0.015739,0.111274,2.606827
5,"frozenset({babies food, cooking oil})",frozenset({napkins}),0.015739,0.368720,2.606827
6,"frozenset({dog food, napkins})",frozenset({cooking oil}),0.015843,0.350689,2.573115
7,frozenset({cooking oil}),"frozenset({dog food, napkins})",0.015843,0.116244,2.573115
8,frozenset({energy drink}),frozenset({airpods}),0.013076,0.209767,2.517290
9,frozenset({airpods}),frozenset({energy drink}),0.013076,0.156912,2.517290


In [44]:
#customer_merged = pd.merge(
 #   customer_basket, 
  #  customer[['customer_id', 'final_cluster_label']], 
   # on='customer_id', 
    #how='inner')
#customer_merged

Note: some baskets were lost due to outlier removal.


**Temos q ver como fazemos com isto, se so aceitamos ou corremos os notebooks todos com os outliers mm**